# 5. Data governance: licences, land, Indigenous data: exercises

Exercises of [chapter 5](../../05_data_governance.md). Run the setup cell first. The exercises change a *copy* of the sample's site registry, never the real one.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, tempfile
import yaml
from data_acquisition.catalog import build, descriptors
from data_acquisition.catalog import summary as summaries
from data_acquisition.readers import eml
from data_acquisition.steps.curate import curate_item

REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())
MY_REGISTRY = Path(tempfile.mkdtemp()) / "sites.yaml"
MY_REGISTRY.write_text((SAMPLE / "sites.yaml").read_text())

**1. Licences from records.** Read the sample's `metadata.xml` with `eml.read` and print its licence. Change the licence URL in the text to CC BY-SA 4.0, then to a licence the reader does not know: what does it return?

> **From the chapter.** §1: each record format maps licences to SPDX identifiers, with its own table.
>
> **Code.** `eml.read` and its table `eml.LICENCES`, in [`data_acquisition/readers/eml.py`](../../../../data_acquisition/readers/eml.py).

In [ ]:
text = (DATASET / "metadata.xml").read_text()
...  # your code here

**2. A registry entry.** In `MY_REGISTRY` (a copy of the sample's registry), set `land_status` to `{value: ..., by: <you>, date: <today>, note: ...}` and check it with `summaries.check_sites`. Then try an entry without `by`.

> **From the chapter.** §4: a manual field in the registry is either `null` or `{value, by, date[, note]}`.
>
> **Code.** `summaries.check_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `MY_REGISTRY` is defined above: a copy of the sample's registry.

In [ ]:
registry = yaml.safe_load(MY_REGISTRY.read_text())
...  # your code here

**3. From registry to items.** Build and curate the sample with your registry (point `BEAVER_SITES` to it). What does the orthomosaic item say about `land_status`, and where did the value come from (`beaver:origin`)? What changes if `by` is `claude`?

> **From the chapter.** §4: `curate` copies the registry into the items and records who entered each value. `beaver:origin` is described in [chapter 4](../../04_catalogs_and_metadata.md), §2.
>
> **Code.** `summaries.load` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), and `curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py).

In [ ]:
os.environ["BEAVER_SITES"] = str(MY_REGISTRY)
...  # your code here
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")      # back to the sample's registry

**4. Hints only.** Add `land_status: private` under the site in a copy of the reference summary, next to `hints`. What does the check say?

> **From the chapter.** §4: summaries may only suggest values for a site.
>
> **Code.** `summaries.check` and `summaries.load_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `REFERENCE` is defined above: the reference summary, as written.

In [ ]:
s = copy.deepcopy(REFERENCE)
...  # your code here (summaries.check(s, summaries.load_sites()))

**5. Find it in the code.** Where does `curate` copy the registry's fields into items, and where does a site collection's licence become `various`?

> **From the chapter.** §4 for what is copied, §1 for the collection's licence.
>
> **Code.** Look in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py) and [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py).

In [ ]:
import inspect
...  # your code here